# Notebook Colab (T4) — RAG Formulaire

Ce notebook prépare un environnement Colab T4 pour tester le pipeline RAG sur les formulaires IRCC en français. Il permet de :

- Vérifier le GPU disponible et configurer le dépôt.
- Installer les dépendances et construire un petit index.
- Poser des questions sans passer par la CLI interactive.

> **Astuce :** utilisez un quota réduit de formulaires (ex. 30) pour accélérer l'ingestion sur Colab.

> **Note :** Ce notebook utilise maintenant le code intégré directement depuis le dépôt. Les patches précédents ont été intégrés dans les modules source.

## 1) Vérifier le GPU

In [ ]:
!nvidia-smi

## 2) Préparer le dépôt

- Définissez `RAG_FORM_REPO_URL` si le dépôt n'est pas déjà présent dans `/content/rag-formulaire`.
- Le notebook ajoute automatiquement le dépôt au `PYTHONPATH` pour l'installation en mode développement.

In [ ]:
import os
import pathlib
import sys

REPO_URL = os.environ.get("RAG_FORM_REPO_URL", "").strip()
REPO_URL = "https://github.com/abdelmajidlra/rag-formulaire.git"
WORKDIR = pathlib.Path("/content/rag-formulaire")

if not WORKDIR.exists():
    if not REPO_URL:
        raise ValueError(
            "Définissez RAG_FORM_REPO_URL ou clonez le dépôt dans /content/rag-formulaire avant d'exécuter ce notebook."
        )
    else:
        print(f"Clonage du dépôt depuis {REPO_URL}…")
        get_ipython().system(f"git clone {REPO_URL} {WORKDIR}")

get_ipython().run_line_magic("cd", str(WORKDIR))
if str(WORKDIR) not in sys.path:
    sys.path.append(str(WORKDIR))

# Add the 'src' directory to sys.path for direct module imports
SRC_DIR = WORKDIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

## 3) Installer les dépendances

L'installation en mode développement (`-e .`) permet de modifier le code localement pendant la session Colab.

In [ ]:
get_ipython().system("pip -q install -U pip setuptools wheel")
get_ipython().system("pip -q install -e .")
!pip install -q bitsandbytes

## 4) Paramétrage rapide

Vous pouvez ajuster les variables pour contrôler la taille de l'ingestion et activer/désactiver GraphRAG.
- `RAG_FORM_MIN_FORMS`: nombre minimum de formulaires (les formulaires synthétiques complètent si besoin).
- `RAG_FORM_MAX_SYNTH`: nombre maximal de formulaires synthétiques générés.
- `RAG_FORM_BASE_DIR`: dossier racine où les données (`data/`) seront écrites.

In [ ]:
from pprint import pprint

os.environ.setdefault("RAG_FORM_BASE_DIR", str(WORKDIR))
os.environ.setdefault("RAG_FORM_MIN_FORMS", "30")
os.environ.setdefault("RAG_FORM_MAX_SYNTH", "0")
os.environ.setdefault("RAG_FORM_ENABLE_GRAPHRAG", "false")

print("Configuration en cours :")
pprint({k: os.environ[k] for k in sorted(os.environ) if k.startswith("RAG_FORM_")})

## 5) Construire l'index (BM25 + vecteur)

Cette étape télécharge les formulaires, découpe les documents puis construit les index. Ajustez `min_forms` pour accélérer sur Colab.

> **Optimisations intégrées:**
> - LLM Singleton: Une seule instance du modèle est chargée (économise ~50% de mémoire)
> - Downloader Deduplication: Évite les doublons dans le manifest
> - Smart Retrieval: Détection automatique des codes de formulaire spécifiques

In [ ]:
from rag_formulaire.ingest import ingest_pipeline

index_store = ingest_pipeline(min_forms=int(os.environ["RAG_FORM_MIN_FORMS"]))
print(f"Chunks indexés : {len(index_store.chunk_map)}")

### Aperçu du manifest

In [ ]:
import json
from rag_formulaire import config

with open(config.MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Formulaires disponibles : {len(manifest)}")
for entry in manifest[:3]:
    print(entry)

## 6) Poser des questions (sans CLI)

Le bloc suivant instancie les composants du pipeline et expose une fonction `ask_question` pour tester rapidement vos requêtes en français.

In [ ]:
from rag_formulaire import config
from rag_formulaire.evaluation import AdvancedSelfReflector, CRAGEvaluator, verify_response_against_evidence
from rag_formulaire.indexing import load_indexes
from rag_formulaire.llm import LocalLLM
from rag_formulaire.query_processing import AgenticQueryRouter, MultilingualQueryHandler, QueryDecomposer, QueryExpander
from rag_formulaire.reranker import CrossEncoderReranker
from rag_formulaire.retrieval import HybridRetriever

config.CHUNK_SIZE = 200
config.CHUNK_OVERLAP = 30  # Petit chevauchement pour garder le contexte

index_store = load_indexes()
query_handler = MultilingualQueryHandler()
router = AgenticQueryRouter()
expander = QueryExpander()
decomposer = QueryDecomposer()
retriever = HybridRetriever(index_store)
reranker = CrossEncoderReranker()
evaluator = CRAGEvaluator()
reflector = AdvancedSelfReflector()
llm = LocalLLM()

In [ ]:
def ask_question(question: str, evidence_k: int = config.FINAL_EVIDENCE_K):
    q_orig, q_fr = query_handler.normalize(question)
    route = router.route(q_fr)
    expansions = expander.expand(q_fr, n=3)
    subqueries = decomposer.decompose(q_fr) if route == "MULTI_STEP" else [q_fr]

    candidates = []
    for sub in subqueries:
        for variant in expansions:
            candidates.extend(retriever.retrieve(variant, manifest=None))

    reranked = reranker.rerank(q_fr, candidates, top_n=config.RERANK_TOP_N)
    if not reranked:
        return {"route": route, "answer": "Aucun extrait trouvé.", "evidence": []}

    scores = list(range(len(reranked), 0, -1))
    if not evaluator.is_evidence_strong(scores, reranked):
        return {"route": route, "answer": evaluator.fallback_message(), "evidence": []}

    evidence_texts = [
        f"[{c.base_chunk.form_code}] {c.base_chunk.section_title}: {c.base_chunk.content}"
        for c in reranked[:evidence_k]
    ]
    system_prompt = (
        "Vous êtes un assistant spécialisé dans les formulaires IRCC. Répondez uniquement en français en vous basant sur les "
        "extraits fournis. Citez le code du formulaire et la section."
    )
    user_prompt = q_fr + "Extraits:" + "".join(evidence_texts)
    answer = llm.chat(system_prompt, user_prompt, max_new_tokens=256)

    if verify_response_against_evidence(answer, reranked):
        answer = reflector.reflect(q_fr, answer, reranked)
    else:
        answer = evaluator.fallback_message()

    return {
        "route": route,
        "expansions": expansions,
        "answer": answer,
        "evidence": reranked[:evidence_k],
    }

### Utilitaires d'affichage

Fonction pour afficher les résultats de manière formatée avec Markdown.

In [ ]:
from IPython.display import display, Markdown

def display_result(result, manifest_list=None):
    """
    Affiche la réponse et les sources de manière formatée en Markdown.
    """
    # 1. En-tête avec la route utilisée
    md = f"### 🤖 Réponse (Stratégie : `{result['route']}`)\n\n"

    # 2. La réponse générée
    md += f"{result['answer']}\n\n"

    # 3. Les sources (Preuves)
    md += "---\n#### 🔍 Sources utilisées :\n"

    # Création d'un dictionnaire pour retrouver les URL à partir du code formulaire
    url_map = {m['form_code']: m['pdf_url'] for m in manifest_list} if manifest_list else {}

    for i, ev in enumerate(result['evidence'], 1):
        chunk = ev.base_chunk
        form_code = chunk.form_code
        section = chunk.section_title

        # Lien vers le PDF officiel si disponible
        if form_code in url_map:
            source_link = f"[{form_code}]({url_map[form_code]})"
        else:
            source_link = f"**{form_code}**"

        # Petit extrait du texte pour contexte (nettoyé des sauts de ligne)
        preview = chunk.content.replace("\n", " ")[:500] + "..."

        md += f"{i}. {source_link} — *{section}* (Page {chunk.page_number})\n"
        md += f"   > <small>{preview}</small>\n"

    display(Markdown(md))

### Gestion de la mémoire GPU

Outils pour nettoyer le cache GPU entre les requêtes et éviter les erreurs OOM (Out Of Memory).

In [ ]:
import torch
import gc

# 1. Force Python garbage collection
gc.collect()

# 2. Clear CUDA (GPU) cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ GPU cache cleared.")

# 3. Check status
!nvidia-smi

## 7) Tests avec questions variées

Testez le pipeline avec une série de questions pour valider le système.

> **Note:** Le système détectera automatiquement les codes de formulaire spécifiques (comme "IMM 5476") et filtrera les résultats en conséquence.

In [ ]:
import torch
import gc

# Liste de questions pour couvrir le manifest
test_questions = [
    # --- Identification des formulaires ---
    "À quoi sert le formulaire IMM 0008 Annexe 9 ?",
    "Quel est le code du formulaire pour la Déclaration d'union de fait ?",
    "Quel formulaire utiliser pour la déclaration d'un parent d'un mineur durant la COVID-19 ?",
    
    # --- Détails personnels et antécédents ---
    "Quelles maladies sont mentionnées dans le questionnaire sur l'état de santé (IMM 5955) ?",
    "Quel formulaire demande des détails sur les études, l'emploi et le déplacement (IMM 0104) ?",
    
    # --- Représentation et Confidentialité ---
    "Qu'est-ce qu'un représentant rémunéré selon le formulaire IMM 5476 ?",
    "Quel formulaire utiliser pour autoriser la communication de renseignements à une personne désignée ?",
    
    # --- Permis de travail ---
    "Quelle est la liste de contrôle des documents pour un permis de travail ?",
    "Quel formulaire remplir pour une demande de permis de travail présentée à l'extérieur du Canada ?",
    
    # --- Frais et Remboursements ---
    "Comment demander un renvoi des frais de traitement (IMM 5741) ?",
]

print(f"🚀 Lancement de la série de {len(test_questions)} questions avec gestion mémoire...\n")

for i, question in enumerate(test_questions, 1):
    print(f"▶️ Question {i}/{len(test_questions)}: {question}")

    try:
        # Interrogation du pipeline
        result = ask_question(question)

        # Affichage propre
        display_result(result, manifest_list=manifest)

    except RuntimeError as e:
        if "out of memory" in str(e):
            print("⚠️ ERREUR OOM : Mémoire GPU saturée sur cette question.")
        else:
            print(f"⚠️ Erreur inattendue : {e}")

    print("\n" + "="*80 + "\n")

    # --- NETTOYAGE MÉMOIRE CRITIQUE ---
    # 1. Supprimer la référence aux résultats (qui peuvent contenir des tenseurs)
    if 'result' in locals():
        del result

    # 2. Forcer le Garbage Collector Python à libérer la RAM système
    gc.collect()

    # 3. Vider le cache de la mémoire vidéo (VRAM) du GPU
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # ----------------------------------